# requires-grad-leaf-assert — ex1: guard optimizer init with leaf + requires_grad asserts

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `requires-grad-leaf-assert`. Running the final beacon cell reports progress against the `Generative: requires_grad leaf assert` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: requires_grad leaf assert` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`requires-grad-leaf-assert`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "requires-grad-leaf-assert"
DD_SUBTOPIC = "Generative: requires_grad leaf assert"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `requires_grad` + leaf assertion — quick refresher

Optimizers train *leaf parameters* — tensors created by `nn.Parameter(...)` or `t.tensor(..., requires_grad=True)`. A *non-leaf* tensor is one produced by an op (e.g. `param * 2`) — it carries gradients during backward but `opt.step()` won't update it (it has no `.data` storage of its own).

**The defensive pattern.** Before passing a list of params to `Adam([...], lr=...)`, assert:
```python
for p in params:
    assert p.is_leaf, f'{p.shape} is non-leaf — optimizer will silently skip it'
    assert p.requires_grad, f'{p.shape} has requires_grad=False — will not update'
```

**Why this saves hours.** A common bug: you `.to(device)` a single Parameter (instead of the whole Module). The result is a NEW non-leaf tensor that the optimizer accepts, runs `.step()` on without error, and silently does nothing. Training loss plateaus, you spend a day looking at the model — the bug is in the optimizer init.

**Module-level fix.** Build your `nn.Module`, move the WHOLE module with `model.to(device)`, then pass `model.parameters()` to the optimizer. Every yielded tensor is a leaf by construction.

### Exercise 1 — guard optimizer init with leaf + requires_grad asserts

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `assert p.is_leaf and p.requires_grad` defensively over an iterable of would-be optimizer params, raising on the FIRST non-leaf or grad-disabled tensor with a useful message.
> Keywords: leaf, requires_grad, optimizer, defensive-assert
> ```

**KCs targeted:** `assert-p-is-leaf`, `assert-p-requires-grad`

Implement `ex1_assert_optim_ready(params)`. The safety check that saves you from silent-no-op optimizer bugs:

1. `params` is an iterable of tensors that you intend to pass to an optimizer constructor.
2. For each tensor `p`:
   - Assert `p.is_leaf` — non-leaf tensors are op outputs, not trainable variables; `opt.step()` silently skips them.
   - Assert `p.requires_grad` — params with `requires_grad=False` never receive gradient, so the optimizer can't update them.
3. If a param fails EITHER assertion, raise `AssertionError` with a message that names which check failed and includes the tensor shape (so the user can find the offender).
4. If all params pass, return `True`.

Input: list of `Tensor` (possibly `nn.Parameter`).
Output: `True` if all checks pass; otherwise raise `AssertionError`.

The visualization shows a pass/fail grid across a mix of leaf / non-leaf / grad-on / grad-off params so you can see which categories the optimizer would silently skip.

In [ ]:
def ex1_assert_optim_ready(params) -> bool:
    """Assert every param is a grad-tracking leaf, suitable for an optimizer."""
    raise NotImplementedError()


def _test_ex1():
    from torch import nn

    # All-pass case — fresh nn.Parameter list.
    good = [nn.Parameter(t.randn(3)), nn.Parameter(t.randn(2, 2))]
    assert ex1_assert_optim_ready(good) is True, 'all-leaf, requires_grad=True should pass'

    # requires_grad=False case → must raise with a useful message.
    frozen = t.randn(3)               # requires_grad=False by default for raw tensor
    try:
        ex1_assert_optim_ready([frozen])
        raise AssertionError('non-grad tensor must trigger the assertion')
    except AssertionError as e:
        msg = str(e)
        assert ('requires_grad' in msg or 'grad' in msg.lower()), (
            f'error must mention requires_grad / grad — got: {msg!r}'
        )

    # Non-leaf case — produce a non-leaf tensor by doing an op on a leaf.
    leaf_param = nn.Parameter(t.randn(4))
    non_leaf = leaf_param * 2   # non-leaf (result of multiplication)
    assert not non_leaf.is_leaf, 'sanity: non_leaf must indeed be non-leaf'
    assert non_leaf.requires_grad, 'sanity: non_leaf must propagate requires_grad'
    try:
        ex1_assert_optim_ready([non_leaf])
        raise AssertionError('non-leaf tensor must trigger the assertion')
    except AssertionError as e:
        msg = str(e)
        assert ('leaf' in msg.lower() or 'is_leaf' in msg), (
            f'error must mention leaf / is_leaf — got: {msg!r}'
        )

    # Mixed-failure case — the FIRST bad param triggers, downstream params do not run.
    ok = nn.Parameter(t.randn(3))
    bad = t.randn(2)  # requires_grad=False
    trigger_count = {'n': 0}
    class CountingParam:
        # Lightweight wrapper that simulates a third 'param' we should never reach.
        def __init__(self):
            self._real = nn.Parameter(t.randn(1))
        @property
        def is_leaf(self):
            trigger_count['n'] += 1
            return self._real.is_leaf
        @property
        def requires_grad(self):
            return self._real.requires_grad
        @property
        def shape(self):
            return self._real.shape
    never_reached = CountingParam()
    try:
        ex1_assert_optim_ready([ok, bad, never_reached])
        raise AssertionError('mixed list with bad param must raise')
    except AssertionError:
        pass
    # We don't strictly require short-circuit, but it's a hint about implementation.

    # Empty-iterable edge case — vacuously true.
    assert ex1_assert_optim_ready([]) is True, 'empty params list is vacuously fine'

    # Real Module case — model.parameters() should all pass.
    model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
    assert ex1_assert_optim_ready(list(model.parameters())) is True, 'fresh Module params must pass'

    # --- Visualization: 2x2 pass/fail grid over leaf/non-leaf × grad-on/grad-off ---
    cases = {
        'leaf + grad-on':    nn.Parameter(t.randn(3)),
        'leaf + grad-off':   t.randn(3),                                            # requires_grad=False
        'non-leaf + grad-on':  nn.Parameter(t.randn(3)) * 2,                        # op output
        'non-leaf + grad-off': (t.randn(3) * 2).detach(),                           # detached
    }
    results = []
    for label, p in cases.items():
        try:
            ex1_assert_optim_ready([p])
            results.append((label, True))
        except AssertionError:
            results.append((label, False))
    fig, ax = plt.subplots(figsize=(6, 3))
    labels_v = [r[0] for r in results]
    colors = ['#3a8' if r[1] else '#d33' for r in results]
    ax.barh(labels_v, [1] * len(results), color=colors, edgecolor='black')
    for i, (lab, ok) in enumerate(results):
        ax.text(0.5, i, 'PASS' if ok else 'FAIL', ha='center', va='center', color='white', weight='bold')
    ax.set_xlim(0, 1)
    ax.set_xticks([])
    ax.set_title('ex1 leaf + requires_grad guard — only top case is optimizer-safe')
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_assert_optim_ready(params) -> bool:
    for p in params:
        assert p.is_leaf, (
            f'param shape={tuple(p.shape)} is NON-LEAF — optimizer will silently skip it. '
            'Move the whole nn.Module with .to(device), not individual Parameters.'
        )
        assert p.requires_grad, (
            f'param shape={tuple(p.shape)} has requires_grad=False — '
            'no gradient will ever flow, optimizer cannot update it.'
        )
    return True
```

**Why this saves hours.** A common bug: you `.to(device)` a single `nn.Parameter` instead of the whole Module. The result is a NEW non-leaf tensor that's `requires_grad=True` and shares no memory with the original. The optimizer accepts it, `.step()` runs without error, and silently does nothing. Loss plateaus, you debug for a day. This assert catches it at construction.

**`is_leaf` semantics.** A tensor is a leaf if it has no `grad_fn`. Leaves are: (a) tensors you constructed via `nn.Parameter(...)` or `t.tensor(..., requires_grad=True)`, and (b) any tensor with `requires_grad=False`. Non-leaves are op outputs that participate in autograd — they get gradients during backward (visible at `.grad` only if `.retain_grad()`), but `opt.step()` only updates leaves.

**Why two separate asserts.** `requires_grad=True` does NOT imply `is_leaf=True`. A non-leaf can have grad enabled (it almost always does — that's how autograd flows). Both conditions must hold for the optimizer to do useful work.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()